In [1]:
from openai import OpenAI
from dotenv import load_dotenv
from tenacity import (
    retry,
    stop_after_attempt,
    wait_random_exponential
    )
import tiktoken
import os
import re
import json

load_dotenv()

def clean_response(text:str) -> str:
    # Паттерн ищет всё между <think> и </think>, включая переносы строк
    pattern = r"<think>.*?</think>"
    
    # Заменяем найденное на пустоту и убираем лишние пробелы по краям
    cleaned_text = re.sub(pattern, "", text, flags=re.DOTALL).strip()
    return cleaned_text

client = OpenAI(
    api_key=os.getenv('GROQ_TOKEN'),
    base_url=os.getenv('BASE_URL_GROQ')
)

In [2]:
# Часть 1
## Задача 1.1

prompt = '''
Сгенерируй список из 4 ИТ профессий в формате json. В выводе каждой профессии должны быть поля:
- title: "Название должности"
- description: "Краткое описание (1 предложение)
- key_skills: ['навык_1','навык_2','навык_3']
- avg_salary_usd: средняя зарплата в долларах
'''

response = client.chat.completions.create(
    model = os.getenv('GROQ_MODEL'),
    messages=[
        {
            'role':'user',
            'content':prompt
        }
    ],
    response_format={'type':'json_object'}
)

print(response.choices[0].message.content)

data = json.loads(response.choices[0].message.content)

[
{
    "title": "Software Developer",
    "description": " Designs, develops, and maintains software applications.",
    "key_skills": ["Programming languages", "Problem-solving", "Version control"],
    "avg_salary_usd": 100000
  },
  {
    "title": "Data Scientist",
    "description": "Analyzes complex data to extract insights and build predictive models.",
    "key_skills": ["Machine learning", "Statistical analysis", "Python/R"],
    "avg_salary_usd": 115000
  },
  {
    "title": "Network Engineer",
    "description": " Manages and secures computer networks.",
    "key_skills": ["Networking protocols", "Cybersecurity", "Network design"],
    "avg_salary_usd": 95000
  },
  {
    "title": "DevOps Engineer",
    "description": "Automates software development processes and operations.",
    "key_skills": ["CI/CD pipelines", "Cloud computing", "Scripting"],
    "avg_salary_usd": 120000
  }
]


In [3]:
import pandas as pd

df = pd.DataFrame(data)

sorted_df = df.sort_values(by=['avg_salary_usd'], ascending=False)

filtered_df = sorted_df[['title','avg_salary_usd']]

filtered_df

,title,avg_salary_usd
3,DevOps Engineer,120000
1,Data Scientist,115000
0,Software Developer,100000
2,Network Engineer,95000


In [6]:
## Задача 1.2

position_review = '''Компания DataVision открывает вакансию старшего аналитика данных в московском офисе.
Требования: опыт работы от 3 лет, знание Python и SQL обязательно, опыт с Spark будет
плюсом. Зарплата: 180 000 — 220 000 рублей в месяц. Формат работы: гибридный (2 дня в офисе).
Контакт: hr@datavision.ru'''

sys_prompt = '''Тебе будут передавать описание вакансий. Извлеки из текста структурированные данные в json формате.

Ограничения:
Если каких-то данных нет в тексте не выдумывай их а напиши "нет данных"

<Начало блока структуры ответа>
Структура json:
"company": "Наименование компании",
"position": "Позиция на которую нанимают",
"location": "город расположения места работы",
"experience_years": опыт работы в годах (число),
"required_skills": ["требуемые скиллы"],
"optional_skills": ["дополнительные скиллы"],
"salary_min": минимальная зарплата,
"salary_max": максимальная зарплата,
"work_format": "формат работы",
"contact": "контакты для связи"
<Конец блока структуры ответа>'''

response = client.chat.completions.create(
    model=os.getenv('GROQ_MODEL'),
    messages=[
        {
            'role':'system',
            'content':sys_prompt
        },
        {
            'role':'user',
            'content':position_review
        }
    ],
    response_format={'type':'json_object'}
)

print(response.choices[0].message.content)

{  "company": "DataVision",
  "position": "Старший аналитик данных",
  "location": "Москва",
  "experience_years": 3,
  "required_skills": ["Python", "SQL"],
  "optional_skills": ["Spark"],
  "salary_min": 180000,
  "salary_max": 220000,
  "work_format": "гибридный",
  "contact": "hr@datavision.ru"
}


In [11]:
# Часть 2
## Задача 2.1
import openai

def safe_request(prompt:str, api_key:str=None, role:str='user'):
    try:
        client_safe = OpenAI(
            api_key=api_key,
            base_url=os.getenv('BASE_URL_GROQ')
        )

        response = client_safe.chat.completions.create(
            model=os.getenv('GROQ_MODEL'),
            messages=[
                {
                    'role':role,
                    'content':prompt
                }
            ]
        )

        return response.choices[0].message.content

    except openai.AuthenticationError as e:
        print(f'[AUTH ERROR] Неверный API-ключ.')
        pass
    except openai.RateLimitError as e:
        print(f'[RATE LIMIT] Превышен лимит запросов. Попробуйте позже')
        pass
    except openai.BadRequestError as e:
        print(f'[BAD REQUEST] Некорректный запрос: {e}')
        pass
    except Exception as e:
        print(f'[ERROR] Неизвестная ошибка: {e}')
        pass

print("--- Тест 1: корректный запрос ---")
result = safe_request(prompt='Hi', api_key=os.getenv('GROQ_TOKEN'))
print(result)

print("\n--- Тест 2: неверный API-ключ ---")
safe_request(prompt='Hi', api_key='invalid key')

print("\n--- Тест 3: невалидная роль ---")
safe_request(prompt='Hi', api_key=os.getenv('GROQ_TOKEN'), role='god')

--- Тест 1: корректный запрос ---
<think>
Okay, the user said "Hi". That's pretty open-ended. I should respond warmly and ask how I can assist them. Let me make sure to keep it friendly and encouraging so they feel comfortable asking their question. Maybe something like "Hello! How can I help you today?" Yeah, that sounds good. It's concise and inviting.
</think>

Hello! How can I help you today?

--- Тест 2: неверный API-ключ ---
[AUTH ERROR] Неверный API-ключ.

--- Тест 3: невалидная роль ---
[BAD REQUEST] Некорректный запрос: Error code: 400 - {'error': {'message': "'messages.0' : discriminator property 'role' has invalid value", 'type': 'invalid_request_error'}}


In [16]:
## Задача 2.2

def logged_request(prompt:str, api_key:str=None, role:str='user'):
    print(f"[LOG] Отправка запроса... (символов в промпте: {len(prompt)})")

    try:
        client_safe = OpenAI(
            api_key=api_key,
            base_url=os.getenv('BASE_URL_GROQ')
        )

        response = client_safe.chat.completions.create(
            model=os.getenv('GROQ_MODEL'),
            messages=[
                {
                    'role':role,
                    'content':prompt
                }
            ]
        )

        get_content = clean_response(response.choices[0].message.content)


        print(f'[LOG] Ответ получен. Токенов использовано: {response.usage.total_tokens}')

        return get_content

    except Exception as e:
        print(f'[LOG] Запрос завершился ошибкой: {e}')
        pass

print(logged_request(prompt="Расскажи кратко о Python за 2 предложения.", api_key=os.getenv('GROQ_TOKEN')))

[LOG] Отправка запроса... (символов в промпте: 42)
[LOG] Ответ получен. Токенов использовано: 324
Python — это высокоуровневый и простой в освоении язык программирования с синтаксисом, близким к человеческой речи, что делает его популярным среди новичков и профессионалов для решения задач от автоматизации до машинного обучения. Он обладает обширными библиотеками и активным сообществом, что обеспечивает его гибкость и применение в веб-разработке, науке данных, искусственном интеллекте и других сферах.


In [23]:
# Часть 3
## Задача 3.1

products = [
    "iPhone 15 Pro",
    "Samsung Galaxy S24",
    "Google Pixel 9",
    "OnePlus 12"
]

print('=== ВАРИАНТ А: по одному запросу ===')
token_count = 0
for product in products:
    response = client.chat.completions.create(
        model = os.getenv('GROQ_MODEL'),
        messages=[
            {
                'role':'system',
                'content':'''Сгенерируй ответ - описание смартфона по его наименованию. Описание должно содержать: бренд, ОС и примерную цену в USD. Не выводи больше никакой дополнительной информации кроме тех пунктов которые описаны выше. 
                
Структура ответа:
Бренд: OnePlus  
ОС: Android 13  
Цена: от $800'''
            },
            {
                'role':'user',
                'content':product
            }
        ]
    )

    print(f'===> Описание продукта: {product} <===')
    get_content = clean_response(response.choices[0].message.content)
    print(f'\n{get_content}\n')
    token_count += response.usage.total_tokens
print(f'\nСуммарно токенов: {token_count}')


print('=== ВАРИАНТ B: батчинг ===')
response = client.chat.completions.create(
    model = os.getenv('GROQ_MODEL'),
    messages=[
        {
            'role':'system',
            'content':'''Тебе передают список смартфонов. Для каждого укажи: бренд, ОС и примерную цену в USD. Отвечай по каждому отдельной строкой. 
            
            Структура ответа:
Бренд: OnePlus  
ОС: Android 13  
Цена: от $800'''
        },
        {
            'role':'user',
            'content':', '.join(products)
        }
    ]
)

get_content = clean_response(response.choices[0].message.content)
tokens = response.usage.total_tokens

print(get_content)
print(f'Суммарно токенов: {tokens}')

print('\n=== СРАВНЕНИЕ ===')
print(f'Батчинг сэкономил {token_count - tokens} токенов ({round((token_count - tokens) * 100/token_count, 2)}%)')

=== ВАРИАНТ А: по одному запросу ===
===> Описание продукта: iPhone 15 Pro <===

Бренд: Apple  
ОС: iOS 17  
Цена: от $999

===> Описание продукта: Samsung Galaxy S24 <===

Бренд: Samsung  
ОС: Android 14  
Цена: от $800

===> Описание продукта: Google Pixel 9 <===

Бренд: Google  
ОС: Android 15  
Цена: от $750

===> Описание продукта: OnePlus 12 <===

Бренд: OnePlus  
ОС: Android 13  
Цена: от $800


Суммарно токенов: 1632
=== ВАРИАНТ B: батчинг ===
Бренд: Apple  
ОС: iOS 17  
Цена: от $999  

Бренд: Samsung  
ОС: Android 14  
Цена: от $799  

Бренд: Google  
ОС: Android 14  
Цена: от $799  

Бренд: OnePlus  
ОС: Android 13  
Цена: от $800
Суммарно токенов: 568

=== СРАВНЕНИЕ ===
Батчинг сэкономил 1064 токенов (65.2%)


In [31]:
## Задача 3.2

system_prompt = '''Тебе передают список смартфонов. Предоставь ответ в виде json. Поля в json: бренд, ОС и примерная цена в USD.'''

response = client.chat.completions.create(
    model = os.getenv('GROQ_MODEL'),
    messages=[
        {
            'role':'system',
            'content':system_prompt
        },
        {
            'role':'user',
            'content':', '.join(products)
        }
    ],
    response_format={'type':'json_object'}
)

get_content = response.choices[0].message.content
df = pd.DataFrame(json.loads(get_content))
print(df)

     бренд       ОС  примерная цена в USD
0    Apple      iOS                  1000
1  Samsung  Android                   700
2   Google  Android                   600
3  OnePlus  Android                   600


In [34]:
# Часть 4
## Задача 4.1

from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
    before_sleep
)

def log_retry(retry_state):
    print(f'[RETRY] Попытка #{retry_state.attempt_number}...')

@retry(
    retry=retry_if_exception_type((openai.RateLimitError, openai.APIConnectionError)),
    stop=stop_after_attempt(4),
    wait=wait_exponential(min=1, max=30),
    before_sleep=log_retry
)
def resilient_request(prompt):
    response = client.chat.completions.create(
        model=os.getenv('GROQ_MODEL'),
        messages=[
            {'role':'user','content':prompt}
        ]
    )

    return clean_response(response.choices[0].message.content)

result = resilient_request("Назови 3 языка программирования одной строкой.")
print(result)

[RETRY] Попытка #1...
[RETRY] Попытка #2...
[RETRY] Попытка #3...


RetryError: RetryError[<Future at 0x159ed0d40 state=finished raised APIConnectionError>]

In [37]:
## Задача 4.2

def log_retry(retry_state):
    print(f'[RETRY] Попытка #{retry_state.attempt_number}...')

@retry(
    retry=retry_if_exception_type((openai.RateLimitError, openai.APIConnectionError)),
    stop=stop_after_attempt(4),
    wait=wait_exponential(min=1, max=30),
    before_sleep=log_retry
)
def production_request(prompt:str, api_key:str=None, role:str='user'):
    try:
        client = OpenAI(
            api_key=api_key,
            base_url=os.getenv('BASE_URL_GROQ')
        )

        response = client.chat.completions.create(
            model=os.getenv('GROQ_MODEL'),
            messages=[
                {'role':role,'content':prompt}
            ]
        )

        return clean_response(response.choices[0].message.content)
    except openai.AuthenticationError as e:
        print(f'[AUTH ERROR] Неверный API-ключ.')
        pass
    except openai.BadRequestError as e:
        print(f'[BAD REQUEST] Некорректный запрос: {e}')
        pass
    except Exception as e:
        print(f'[ERROR] Неизвестная ошибка: {e}')
        pass

result = production_request(
    prompt="Назови 3 языка программирования одной строкой.",
    api_key=os.getenv('GROQ_TOKEN'))
print(result)

result = production_request("", )
print(result)

result = production_request(
    "", 
    api_key=os.getenv('GROQ_TOKEN'),
    role='god')
print(result)

Python, JavaScript, C++.
[ERROR] Неизвестная ошибка: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable
None
[BAD REQUEST] Некорректный запрос: Error code: 400 - {'error': {'message': "'messages.0' : discriminator property 'role' has invalid value", 'type': 'invalid_request_error'}}
None


In [38]:
# Часть 5
## Задача 5.1

prompts = {
    "Короткий": "Привет!",
    "Средний": "Напиши краткое резюме для специалиста по данным с опытом 5 лет в Python и машинном обучении.",
    "Длинный": "Ты опытный технический интервьюер. Задай 10 вопросов кандидату на позицию Senior Data Scientist, охватив темы: статистика, машинное обучение, программирование на Python, работа с большими данными и лидерские навыки. После каждого вопроса укажи, какую компетенцию он проверяет."
}

import tiktoken
import pandas as pd

encoder = tiktoken.encoding_for_model('gpt-4o-mini')

answer_list_dict = []

for prompt in prompts:
    length = len(prompts[prompt])
    tokens = len(encoder.encode(prompts[prompt]))
    symbols_on_token = length / tokens

    answer_list_dict.append(
        {
            'Промпт':prompt,
            "Символов":length,
            'Токенов':tokens,
            "Символов\\токен":symbols_on_token
        }
    )

df = pd.DataFrame(answer_list_dict)
print(df)

     Промпт  Символов  Токенов  Символов\токен
0  Короткий         7        3        2.333333
1   Средний        92       26        3.538462
2   Длинный       276       68        4.058824


In [39]:
## Задача 5.2

def check_and_send(prompt:str, max_tokens:int=500):
    tokens = len(encoder.encode(prompt))

    if tokens > max_tokens:
        print(f'''[WARNING] Промпт содержит {tokens} токенов, превышает лимит {max_tokens}. Запрос не отправлен.''')
    else:
        response = client.chat.completions.create(
            model = os.getenv('GROQ_MODEL'),
            messages = [
                {
                    'role':'user',
                    'content':prompt
                }
            ]
        )

        get_content = clean_response(response.choices[0].message.content)

        print(f'''[OK] Промпт содержит XXX токенов. Отправляю запрос...
[ОТВЕТ] {get_content}''')
        
short_prompt = "Назови столицу России."
long_prompt = "Напиши " + "очень длинный промпт это прям очень длинный промпт" * 50

check_and_send(short_prompt, max_tokens=500)
print()
check_and_send(long_prompt, max_tokens=500)

[OK] Промпт содержит XXX токенов. Отправляю запрос...
[ОТВЕТ] Столицей современной России является **Москва**. С 1918 года она официально является столицей, хотя ранее (до 1918) столицей был Санкт-Петербург. Москва также была столицей в период с 1479 по 1712 год, а после революции 1917 года вновь вернула себе этот статус.

[WARNING] Промпт содержит 702 токенов, превышает лимит 500. Запрос не отправлен.


In [50]:
# Задача 6

reviews = [
    "This book changed my life! The author explains machine learning concepts in a way that's both accessible and deep. Highly recommend to anyone in data science.",
    "Disappointing. The examples are outdated and the explanations are too shallow. Expected much more from this title.",
    "Good introduction, but lacks depth on neural networks. Decent for beginners, frustrating for intermediate readers."
]

def check_num_tokens(prompt:str, max_tokens:int=300) -> tuple:
    tokens = len(encoder.encode(prompt))
    return tokens <= max_tokens, tokens

def translate_review(prompt:str) -> str:
    system_prompt = 'Переведи текст на русский язык.'
    response = client.chat.completions.create(
        model = os.getenv('GROQ_MODEL'),
        messages = [
            {
                'role':'system',
                'content':system_prompt
            },
            {
                'role':'user',
                'content':prompt
            }
        ]
    )
    return clean_response(response.choices[0].message.content)

def classification(prompt:str) -> str:
    system_prompt = 'Классифицируй отзыв. Формат классификации json. Структура результата: {"sentiment":"positive|neutral|negative", "score": 1-5}'

    response = client.chat.completions.create(
        model = os.getenv('GROQ_MODEL'),
        messages = [
            {
                'role':'system',
                'content':system_prompt
            },
            {
                'role':'user',
                'content':prompt
            }
        ],
        response_format = {'type':'json_object'}
    )

    return response.choices[0].message.content

for_analytics = []

for i,review in enumerate(reviews):
    print(f'=== ОБРАБОТКА ОТЗЫВА {i+1} ===')
    try:
        check_tokens = check_num_tokens(review)
        if check_tokens[0]:
            print(f'[OK] Проверка токенов пройдена ({check_tokens[1]} токенов)')
            ru_review = translate_review(review)
            print(f'Перевод: {ru_review}')
            sentiment = classification(ru_review)
            print(f'Настроение: {sentiment}')
            for_analytics.append(f'Кол-во токенов: {check_tokens[1]}, настроение: {sentiment}')
        else:
            print('Кол-во токенов в запросе превышает допустимый лимит')
    except Exception as e:
        print(e)
        pass

print('=== ИТОГ ===')
response = client.chat.completions.create(
    model = os.getenv('GROQ_MODEL'),
    messages = [
        {
            'role':'system',
            'content':'''Проанализируй переданные данные и выведи отчет. Структура отчета: 
            Обработано: соколько обращений обработано из скольки (пример 3/3)
            Positive: X | Neutral: X | Negative: X
            Средняя оценка: X.X/5
            Всего токенов использовано: XXX'''
        },
        {
            'role':'user',
            'content':'. '.join(for_analytics)
        }
    ]
)

print(clean_response(response.choices[0].message.content))

=== ОБРАБОТКА ОТЗЫВА 1 ===
[OK] Проверка токенов пройдена (29 токенов)
Перевод: Эта книга изменила мою жизнь! Автор объясняет понятия машинного обучения доступно и глубоко. Высоко рекомендую всем, кто занимается наукой о данных.
Настроение: {"sentiment":"positive","score":5}
=== ОБРАБОТКА ОТЗЫВА 2 ===
[OK] Проверка токенов пройдена (22 токенов)
Перевод: Разочаровывающе. Примеры устаревшие, а объяснения слишком поверхностные. От этой книги ожидал гораздо большего.
Настроение: {"sentiment":"negative", "score": 2}
=== ОБРАБОТКА ОТЗЫВА 3 ===
[OK] Проверка токенов пройдена (20 токенов)
Перевод: Хорошее введение, но не хватает глубины по нейросетям. Подходит для начинающих, разочаровывает средних читателей.
Настроение: {"sentiment":"negative", "score":3}
=== ИТОГ ===
Обработано: 3/3  
Positive: 1 | Neutral: 0 | Negative: 2  
Средняя оценка: 3.3/5  
Всего токенов использовано: 71
